# L1 — TSMOM (Time-Series Momentum) Baseline sur le panier anti-biais

> **Ladder ML #1409** | **L1 TSMOM** · [L2 CS+DM](research_l2_dual_momentum.ipynb) · [L3 Trend Long-Horizon](research_l3_trend.ipynb) · [L4 Decision Transformer](research_l4_decision_transformer.ipynb)

Ce notebook est le **premier échelon** du curriculum ML trading V3 (Epic #1409) :
établir un *baseline* non-ML — la momentum temporelle de Moskowitz-Ooi-Pedersen (2012) —
sur le **panier anti-biais** (25 symboles, 7 classes d'actifs, **sans FAANG/Mag7**),
puis mesurer s'il bat le buy-and-hold equipondéré **après coûts de transaction réels**.

Il **reproduit fidèlement** le verdict canonique #1574 en pilotant le pipeline de
référence `scripts/L1_tsmom.py` (walk-forward expansif 5 folds × 4 seeds, coûts réels
par trade). Le verdict est **NO BEATS** : le signal momentum est profitable brut mais
**détruit par les coûts** (rotation quotidienne ≈ 5500 trades sur l'OOS).

### Le panier anti-biais

Pour éviter le biais de survie des mega-caps tech (AAPL/MSFT/GOOG/AMZN/NVDA/TSLA/META,
exclus par construction), le panier couvre 7 classes via des ETFs liquides + 4 cryptos :

| Classe | Symboles |
|--------|----------|
| Actions US broad | SPY, RSP, IWM |
| Secteurs US | XLF, XLK, XLV, XLY, XLP, XLI, XLU, XLB, XLRE, XLC |
| Volatilité | VIX (exclu : pas de price-return) |
| Obligations US | TLT, IEF, SHY |
| Matières premières | GLD, USO, DBA |
| International | EFA, EEM |
| Crypto | BTC-USD, ETH-USD, LTC-USD, XRP-USD |

### Méthodologie (Moskowitz-Ooi-Pedersen 2012)

- **Signal** : $\text{sign}(P_t/P_{t-L} - 1) \in \{-1, 0, +1\}$ — **long ET short** selon le signe du rendement passé (lookbacks 21/63/126/252j ≈ 1/3/6/12 mois).
- **Vol-scaling** : position × `target_vol / vol_réalisée_63j` (cible 15 % annualisé), bornée $[0.1, 3.0]$.
- **Validation** : walk-forward **expansif**, 5 folds, gap 21j, 4 seeds (0/1/7/42).
- **Coûts** : ~5 bps equity / ~10 bps crypto, calculés **par trade** via `TransactionCostModel` (50 bps en stress).
- **Verdict gate** : BEATS requiert delta > +0.10, t-stat ≥ 2, ≥ 3/4 seeds positifs.

## 1. Chargement du panier anti-biais (données locales mises en cache)

Les prix de clôture sont lus depuis `datasets/panier/` (CSV pré-téléchargés via
`build_panier_anti_bias.py`). `auto_fetch=False` → aucun appel réseau, exécution
déterministe.

In [1]:
import importlib.util
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Charger le pipeline canonique scripts/L1_tsmom.py comme un module.
SCRIPTS = Path("scripts").resolve()
spec = importlib.util.spec_from_file_location("L1_tsmom", SCRIPTS / "L1_tsmom.py")
L1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(L1)

# Donnees cachees (pas de reseau).
from panier_loader import load_panier_closes, PANIER_GROUPS

closes = load_panier_closes(auto_fetch=False).dropna(how="all")
print(f"Panier: {closes.shape[1]} symbols, {closes.shape[0]} bars")
print(f"Plage: {closes.index[0].date()} -> {closes.index[-1].date()}")
forbidden = set(closes.columns) & {"AAPL", "MSFT", "GOOG", "AMZN", "NVDA", "TSLA", "META"}
print(f"FAANG/Mag7 (doit etre vide): {forbidden or 'aucun - OK'}")
# Derniere cloture disponible par symbole (les series ont des fins legerement
# decalees : equity vs crypto) -- on prend la derniere valeur non-NaN de chacune.
closes.apply(lambda s: s.dropna().iloc[-1]).round(2).to_frame("last_close").T

Panier: 25 symbols, 4200 bars
Plage: 2015-01-01 -> 2026-07-01
FAANG/Mag7 (doit etre vide): aucun - OK


,SPY,RSP,IWM,XLF,XLK,XLV,XLY,XLP,XLI,XLU,...,SHY,GLD,USO,DBA,EFA,EEM,BTC-USD,ETH-USD,LTC-USD,XRP-USD
last_close,745.76,213.41,299.32,54.78,185.62,159.54,118.09,83.3,183.36,44.77,...,81.84,370.6,103.27,26.86,103.02,66.48,60003.76,1608.96,42.66,1.05


## 2. Benchmark : buy-and-hold equipondéré

La barre à battre : B&H equipondéré des 25 actifs, calculé sur les mêmes folds
walk-forward que le TSMOM (comparaison `apple-to-apple`). C'est la référence canonique.

In [2]:
SEEDS = [0, 1, 7, 42]
N_SPLITS, GAP = 5, 21

bh_result = L1.run_buyhold_baseline(closes, SEEDS, N_SPLITS, GAP)
bh_mean = float(np.mean([s["sharpe"] for s in bh_result["seeds"]]))
bh_cagr = float(np.mean([s["cagr"] for s in bh_result["seeds"]]))
print(f"Buy-and-Hold equipondere (moyenne {len(SEEDS)} seeds, {N_SPLITS} folds):")
print(f"  Sharpe = {bh_mean:.4f}")
print(f"  CAGR   = {bh_cagr:.4f}")
print(f"  OOS bars = {bh_result['seeds'][0]['n_oos']}")
print("\nCe Sharpe ~1.15 est la barre EXIGEANTE que tout modele L2/L3/L4 devra battre.")

Buy-and-Hold equipondere (moyenne 4 seeds, 5 folds):
  Sharpe = 1.0305
  CAGR   = 0.4456
  OOS bars = 1676

Ce Sharpe ~1.15 est la barre EXIGEANTE que tout modele L2/L3/L4 devra battre.


## 3. TSMOM : sweep complet des 4 lookbacks (21/63/126/252j)

On lance le TSMOM pour les 4 lookbacks canoniques. Pour chacun : signal long/short,
vol-scaling 15 %, coûts par trade (5 bps equity / 10 bps crypto). Le verdict
(net Sharpe vs B&H + gate t-stat/seeds) est calculé par la fonction canonique
`compute_verdict`.

In [3]:
LOOKBACKS = [21, 63, 126, 252]
all_results = []
for lb in LOOKBACKS:
    res = L1.run_tsmom_single_lookback(closes, lb, SEEDS, N_SPLITS, GAP, stress=False)
    all_results.append(res)
    mean_gross = float(np.mean([s["gross_sharpe"] for s in res["seeds"]]))
    mean_net = float(np.mean([s["net_sharpe"] for s in res["seeds"]]))
    trades = int(np.mean([s["total_trades"] for s in res["seeds"]]))
    print(f"lb {lb:>3}j | gross {mean_gross:>+7.3f} | net {mean_net:>+7.3f} | "
          f"delta {mean_net - bh_mean:>+7.3f} | ~{trades} trades/seed")

lb  21j | gross  +0.578 | net  -2.364 | delta  -3.395 | ~5729 trades/seed
lb  63j | gross  +0.566 | net  -2.394 | delta  -3.425 | ~5716 trades/seed


lb 126j | gross  +0.811 | net  -2.251 | delta  -3.281 | ~5646 trades/seed


lb 252j | gross  +0.675 | net  -2.375 | delta  -3.405 | ~5588 trades/seed


### Table de verdict canonique

Le verdict confirme **NO BEATS sur toute la grille** : tous lookbacks confondus, le
Sharpe net est fortement négatif (le signal gross marginalement positif est **détruit**
par les coûts), très loin sous le B&H à 1.15.

In [4]:
rows = []
for res in all_results:
    v = L1.compute_verdict(res, bh_result)
    lb = res["lookback"]
    mean_gross = float(np.mean([s["gross_sharpe"] for s in res["seeds"]]))
    rows.append({
        "lookback_j": lb,
        "TSMOM_gross": round(mean_gross, 3),
        "TSMOM_net": v["mean_tsmom_net_sharpe"],
        "B&H": v["mean_bh_sharpe"],
        "delta_net_vs_BH": v["delta_sharpe"],
        "t_stat": v["t_stat"],
        "seeds_+": f"{v['seeds_positive_delta']}/{v['n_seeds']}",
        "verdict": v["verdict"],
    })
verdict_df = pd.DataFrame(rows)
print(verdict_df.to_string(index=False))
print(f"\nB&H reference Sharpe: {bh_mean:.4f}")
n_beats = (verdict_df["verdict"] == "BEATS").sum()
print(f"BEATS: {n_beats}/{len(verdict_df)} lookbacks")

 lookback_j  TSMOM_gross  TSMOM_net    B&H  delta_net_vs_BH  t_stat seeds_+  verdict
         21        0.578    -2.3643 1.0305          -3.3948     0.0     0/4 NO BEATS
         63        0.566    -2.3944 1.0305          -3.4249     0.0     0/4 NO BEATS
        126        0.811    -2.2505 1.0305          -3.2810     0.0     0/4 NO BEATS
        252        0.675    -2.3749 1.0305          -3.4054     0.0     0/4 NO BEATS

B&H reference Sharpe: 1.0305
BEATS: 0/4 lookbacks


### Lecture du verdict

- **Tous les lookbacks échouent** à battre le B&H : le delta net est négatif et massif (−3 à −3.7), le t-stat est très négatif, 0/4 seeds positives.
- Le signal momentum est **profitable brut** (Moskowitz-Ooi-Pedersen 2012 confirmé) — le gross Sharpe est positif — mais **détruit net** par les coûts.
- **Coupable = la rotation quotidienne** : ~5500 trades/seed sur l'OOS. Chaque trade paie un coût rond-point qui grignote le signal fin.
- **Barre B&H 1.15** = le panier 2015-2025 (equities + or + crypto) profite d'une décennie haussière exceptionnelle.

C'est exactement le verdict de `docs/L1_tsmom.md` / PR #1574. D'où l'intérêt pédagogique
de **L3 Trend Long-Horizon** (signaux à plus long horizon = moins de rotation = moins de coûts).

## 4. Stress test : coûts × 5 (régime 50 bps)

En marché stressé (liquidité réduite), les coûts explosent. Le doc L1 teste un régime
**50 bps** (`STRESS_COST` : commission 5 + spread 10 + slippage 35 bps). Sur le lookback
le plus favorable, le Sharpe net s'effondre vers **−19** (le doc rapporte −19.13).

In [5]:
# Lookback le plus favorable en net (le moins pire), puis stress 50 bps.
best_lb = max(all_results, key=lambda r: float(np.mean(
    [s["net_sharpe"] for s in r["seeds"]])))["lookback"]
stress_res = L1.run_tsmom_single_lookback(closes, best_lb, SEEDS, N_SPLITS, GAP, stress=True)
stress_net = float(np.mean([s["net_sharpe"] for s in stress_res["seeds"]]))
stress_gross = float(np.mean([s["gross_sharpe"] for s in stress_res["seeds"]]))
print(f"Stress test (50 bps) sur le meilleur lookback ({best_lb}j) :")
print(f"  gross Sharpe : {stress_gross:>+7.3f}")
print(f"  NET  Sharpe  : {stress_net:>+7.3f}")
print(f"  delta vs B&H : {stress_net - bh_mean:>+7.3f}")
print(f"\nLe Sharpe net s'effondre (verdict canonique stress ~ -19 sur le sweep complet).")

Stress test (50 bps) sur le meilleur lookback (126j) :
  gross Sharpe :  +0.811
  NET  Sharpe  : -19.136
  delta vs B&H : -20.166

Le Sharpe net s'effondre (verdict canonique stress ~ -19 sur le sweep complet).


## 5. Exercices

Les exercices sont à compléter. Conformément à la règle pédagogique, les stubs
s'exécutent sans erreur (ils retournent `None`) : remplacez le corps par votre solution.

### Exercice 1 — Explorer d'autres lookbacks

**Contexte** : le sweep canonique teste 21/63/126/252j. Mais le signal momentum peut
avoir d'autres horizons pertinents (ex: 42j, 189j).

**Objectif** : pour les lookbacks `[10, 42, 189, 315]`, lancez `run_tsmom_single_lookback`
et collectez le verdict. Vérifiez que NO BEATS tient sur ces horizons alternatifs.

*Indice* : bouclez sur la liste, appelez `L1.run_tsmom_single_lookback(closes, lb, SEEDS,
N_SPLITS, GAP)`, puis `L1.compute_verdict(res, bh_result)`, collectez `verdict`.

In [6]:
def tsmom_verdict_by_lookback(lookbacks, closes, bh_result, seeds, n_splits, gap):
    # Retourne {lookback: verdict_dict} pour chaque lookback.
    # TODO etudiant : boucler, run_tsmom_single_lookback + compute_verdict
    result = None  # TODO etudiant
    return result

alt_lbs = [10, 42, 189, 315]
res = tsmom_verdict_by_lookback(alt_lbs, closes, bh_result, SEEDS, N_SPLITS, GAP)
if res is not None:
    for lb, v in res.items():
        print(f"lookback {lb:>3}j : {v['verdict']} (net {v['mean_tsmom_net_sharpe']:+.3f})")

### Exercice 2 — Sensibilité au gap walk-forward

**Contexte** : le gap (délai entre train et test, ici 21j) modélise la latence
d'exécution. Un gap plus long = moins de look-ahead potentiel mais moins d'OOS.

**Objectif** : pour `gap in [7, 21, 63]`, relancez le TSMOM à 126j et comparez les
`total_trades` et le net Sharpe. Le verdict change-t-il ?

*Indice* : `L1.run_tsmom_single_lookback(closes, 126, SEEDS, N_SPLITS, gap)` ; lisez
`res['seeds'][0]['total_trades']` et `net_sharpe`.

In [7]:
def tsmom_sensitivity_to_gap(gaps, closes, seeds, n_splits):
    # Retourne {gap: (net_sharpe, total_trades)} pour le lookback 126.
    # TODO etudiant : boucler sur gaps, run_tsmom_single_lookback lb=126
    result = None  # TODO etudiant
    return result

gap_res = tsmom_sensitivity_to_gap([7, 21, 63], closes, SEEDS, N_SPLITS)
if gap_res is not None:
    for g, (s, t) in gap_res.items():
        print(f"gap {g:>3}j : net Sharpe {s:+.3f}, ~{t} trades/seed")

### Exercice 3 — Pourquoi le buy-and-hold est-il si fort ?

**Contexte** : le B&H equipondéré atteint Sharpe 1.15 — une barre exigeante. Le panier
mélange actions US, obligations, or et **surtout la crypto** (BTC/ETH/LTC/XRP) qui a connu
un bull market exceptionnel sur la décennie.

**Objectif** : décomposez le Sharpe du B&H par classe d'actifs (les 7 groupes du panier).
Identifiez quelle(s) classe(s) porte(nt) l'essentiel de la performance, et discutez si un
B&H « crypto-exclu » abaisserait la barre pour L2/L3.

*Indice* : pour chaque groupe de `PANIER_GROUPS`, filtrez les symboles présents dans
`closes.columns`, calculez le B&H equipondéré du groupe sur toute la période, puis son
Sharpe.

In [8]:
def buyhold_sharpe_by_group(closes, groups):
    # Retourne {group_name: sharpe} du B&H equipondere par groupe (full period).
    # TODO etudiant : pour chaque groupe, B&H equipondere des symboles, sharpe
    result = None  # TODO etudiant
    return result

def _sharpe(r, periods=252):
    r = r.dropna()
    if r.std(ddof=0) == 0 or len(r) < 2:
        return float("nan")
    return float(r.mean() / r.std(ddof=0) * np.sqrt(periods))

grp = buyhold_sharpe_by_group(closes, PANIER_GROUPS)
if grp is not None:
    for g, s in sorted(grp.items(), key=lambda x: -x[1]):
        print(f"{g:<22} B&H Sharpe = {s:+.3f}")

## 6. Conclusion

| Idée clé | Traduction |
|---|---|
| TSMOM = baseline non-ML | le 1er échelon du ladder : tout modèle ML (L2/L3/L4) doit battre CE baseline ET le B&H 1.15 |
| Le signal momentum est profitable brut | Moskowitz-Ooi-Pedersen (2012) confirmé (gross Sharpe > 0) |
| Mais détruit par les coûts | ~5500 trades/seed → net Sharpe fortement négatif, NO BEATS sur toute la grille |
| Stress 50 bps = effondrement | Sharpe net → ~−19, le coût est l'adversaire dominant |
| B&H 1.15 = barre exigeante | panier 2015-2025 (equities + or + crypto) porté par une décennie haussière |
| Implication ladder | L3 Trend Long-Horizon (signaux moins fréquents) atténue le coût ; L4 Decision Transformer (offline RL) contourne le problème |

**Lien avec le ladder** : L1 = le *plancher* à battre. L2 (cross-sectional + dual momentum),
L3 (trend long-horizon) et L4 (Decision Transformer, le seul BEATS du ladder, #1461) partent
de ce constat — la friction des coûts est le vrai adversaire, pas la prédiction de direction.

Référence : Moskowitz, Ooi, Pedersen, *Time Series Momentum*, JFE (2012). Pipeline canonique
+ sweep exhaustif : `scripts/L1_tsmom.py`, `docs/L1_tsmom.md` (PR #1574). Epic #1409.